# Calculate CAPs dynamic metrics

This notebook calculates occurrence rate and dwell time of each CAPs and transition probabilities for pairs of CAPs

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

In [2]:
subs = ['Y01', 'Y02', 'Y03', 'Y04', 'Y05', 'Y06', 'Y07', 'Y08', 'Y09','Y10',
        'Y11', 'Y12', 'Y13', 'Y14', 'Y15', 'Y16', 'Y17','Y18', 'Y19', 'Y20']

sessions = [1, 2, 3]
sessions_dict = {1: "Unilateral", 2: "Bilateral", 3: "Sham"}
runs = [2]
runs_dict = {2: "tDCS On"}

load_dotenv()
base_dir = os.environ['BASE_DIR']
runs_str = "run-ON"
output_dir = os.path.join(base_dir, f"coactivation_patterns_{runs_str}")

## Occurrence rate and dwell time 

In [3]:
k=7
run = 2

cluster_data_dir = os.path.join(base_dir, f"coactivation_patterns_{runs_str}", f"caps_nclust-{k}")
out_dir = os.path.join(base_dir, f'caps_dynamics_{runs_str}', f"caps_nclust-{k}")
os.makedirs(out_dir, exist_ok=True)


capdata = np.load(os.path.join(cluster_data_dir, f"CAP_DATA_nclust-{k}.npy"), allow_pickle=True)
capdata = capdata.item()

subject_list = capdata['subject_list']
sessions = capdata['sessions']
runs = capdata['runs']
CAPtemplates = capdata['centers_paired']
CAPtimeseries = capdata['labels_paired'].astype(int)
DATA_LABELS = capdata['DATA_LABELS']
print(capdata.keys())

cap_keys = np.unique(CAPtimeseries)
n_cap = len(cap_keys)

# group-wise occurence rate
occ_rate_group = np.zeros((n_cap))
for i in cap_keys:
    occ_rate_group[i] = sum(CAPtimeseries == cap_keys[i]) / len(CAPtimeseries)

n_subs = len(subject_list)
n_sess = len(sessions)
n_runs = len(runs)

occ_rate_individuals = []
dwelltime_individuals = []
for sub in subject_list:
    for ses in sessions:

        data_label = f"sub-{sub}_ses-{ses}_run-{run}"
        #print(data_label)

        ## OCCURRENCE RATE
        occ_tmp = dict()
        occ_tmp['data_label'] = data_label
        occ_tmp['subject'] = sub
        occ_tmp['session'] = ses
        occ_tmp['run'] = run
        #print(sub, ses, run, data_label)
        sub_cap_data = CAPtimeseries[DATA_LABELS == data_label]
        #print(len(sub_cap_data))
        for i in cap_keys:
            occ_tmp[f'CAP{i + 1}'] = sum(sub_cap_data == cap_keys[i]) / len(sub_cap_data)
        occ_rate_individuals.append(occ_tmp)

        ## DWELL TIME
        dwell_tmp = dict()
        dwell_tmp['data_label'] = data_label
        dwell_tmp['subject'] = sub
        dwell_tmp['session'] = ses
        dwell_tmp['run'] = run
        sub_cap_data = CAPtimeseries[DATA_LABELS == data_label]
        i = 0
        for i in cap_keys:
            a1 = np.diff(np.multiply((sub_cap_data == cap_keys[i]), 1), append=int(sub_cap_data[-1] != cap_keys[i]))
            b1 = np.cumsum(np.multiply((sub_cap_data == cap_keys[i]), 1))
            b2 = b1[a1 == -1]
            try:
                du = np.diff(b2, prepend=b2[0])
            except:
                du = np.zeros(1)
            dwell_tmp[f'CAP{i + 1}'] = np.mean(du)
            dwell_tmp[f'CAP{i + 1}_occurrence_times'] = len(du)
        dwelltime_individuals.append(dwell_tmp)

occ_rate_df = pd.DataFrame(occ_rate_individuals)
occ_rate_df["session_name"] = occ_rate_df["session"].map(sessions_dict)
occ_rate_df["run_name"] = occ_rate_df["run"].map(runs_dict)
occ_rate_df.to_csv(os.path.join(out_dir, f"CAP_occurence_times.csv"), index=False)

dwelltime_df = pd.DataFrame(dwelltime_individuals)
dwelltime_df["session_name"] = dwelltime_df["session"].map(sessions_dict)
dwelltime_df["run_name"] = dwelltime_df["run"].map(runs_dict)
dwelltime_df.to_csv(os.path.join(out_dir, f"CAP_dwelltime.csv"), index=False)

dict_keys(['subject_list', 'sessions', 'runs', 'masker', 'DATA_LABELS', 'labels_raw', 'centers_raw', 'cap_order_paired', 'labels_paired', 'centers_paired', 'centers_average'])


## Transition probabilities

In [4]:
def do_transition_matrix(cap_idx, sub_indices):
    """
    Calculate transition probabilities between CAPs

    Parameters:
    -----------
    cap_idx : numpy array
        1D array containing CAP labels for all timepoints across all subjects
        Example: [0, 0, 1, 2, 2, 2, 1, 3, ...] for k=7 would have values 0-6

    sub_indices : list of lists
        Each element contains [start_idx, end_idx] for a subject's data in cap_idx
        Example: [[0, 379], [379, 758], ...] where each subject has 379 volumes

    Returns:
    --------
    trans_between_cap_times : numpy array (n_cap × n_cap × n_subs)
        Raw count of transitions from CAP_i to CAP_j for each subject

    trans_between_cap_ratio : numpy array (n_cap × n_cap × n_subs)
        Normalized transition probability from CAP_i to CAP_j for each subject
    """

    # Get unique CAP labels (e.g., [0, 1, 2, 3, 4, 5, 6] for k=7)
    cap_keys = np.unique(cap_idx)

    # Number of unique CAPs
    n_cap = len(cap_keys)

    #print(cap_keys)

    # Number of subjects to process (or scans)
    n_subs = len(sub_indices)

    #print(n_cap)
    #print(n_subs)

    #% *between_cap_times: how many times transit from one CAP to another*
    #% *between_cap_times_ratio: num of times (CAP_i to CAP_j) / num of times(CAP_i to the other CAP)
    #% *between_cap_duration_mean: the averaged duration of CAP_i -> CAP_j
    #% *between_cap_duration_std: the std of duration of CAP_i -> CAP_j

    # Initialize output matrices
    # Each subject gets their own n_cap × n_cap transition matrix
    trans_between_cap_times = np.zeros((n_cap, n_cap, n_subs))
    trans_between_cap_ratio = np.zeros((n_cap, n_cap, n_subs))
    trans_between_cap_duration_mean = np.zeros((n_cap, n_cap, n_subs))
    trans_between_cap_duration_std = np.zeros((n_cap, n_cap, n_subs))

    # Process each subject separately
    for s in np.arange(0,n_subs):

        # Extract this subject's CAP sequence using their start:end indices
        cap_idx_sub = cap_idx[sub_indices[s][0]:sub_indices[s][1]]

        # Initialize matrices for this subject
        trans_times = np.zeros((n_cap, n_cap))   # Raw transition counts
        trans_ratio = np.zeros((n_cap, n_cap))   # Normalized probabilities
        trans_du_mean = np.zeros((n_cap, n_cap)) # Mean duration (not used)
        trans_du_std = np.zeros((n_cap, n_cap))  # Std duration (not used)

        # Track how many times each CAP occurs
        occ_times = np.zeros(n_cap)

        # Process each CAP as a source state (where transitions come FROM)
        for i in cap_keys:

            # Create binary mask: 1 where cap_idx_sub equals CAP_i, 0 elsewhere
            # Then take diff to find edges (transitions)
            # prepend=0 ensures we can detect if sequence starts with CAP_i
            a1 = np.diff(np.multiply((cap_idx_sub == cap_keys[i]),1), prepend=0)

            # Find where CAP_i ENDS (transitions OUT)
            # a1 == -1 means we went from 1 to 0 (leaving CAP_i)
            c_idx = (a1 == -1)

            # Calculate cumulative sum of CAP_i occurrences
            # This helps identify distinct episodes of CAP_i
            b1 = np.cumsum(np.multiply((cap_idx_sub == cap_keys[i]),1))

            # Get cumsum values at transition points (end of each CAP_i episode)
            b2 = b1[c_idx]

            # Calculate duration of each CAP_i episode
            try:
                # Difference between consecutive endpoints gives episode durations
                du = np.diff(b2, prepend=b2[0])
            except:
                # If no occurrences, set duration to zero
                du = np.zeros(1)

            # Count how many times CAP_i occurred
            occ_times[i] = len(du)

            # For each possible destination CAP (where transitions go TO)
            for j in cap_keys:

                # Skip self-transitions (staying in same CAP)
                if (i!=j):
                    #print(f"{i}-{j}")

                    # Count transitions: at positions where CAP_i ends (c_idx),
                    # how many times is the next state CAP_j?
                    trans_times[i,j] = np.nansum(cap_idx_sub[c_idx]==j);

                    # Normalize: probability = (i→j transitions) / (total exits from i)
                    trans_ratio[i,j] = trans_times[i,j]/occ_times[i];

                    # These calculate mean/std duration of CAP_i before transitioning to j
                    # Currently not used
                    #trans_du_mean[i,j] = np.nanmean(du[cap_idx_sub[c_idx]==j]);
                    #trans_du_std[i,j] = np.nanstd(du[cap_idx_sub[c_idx]==j]);


        # Store this subject's results in the 3D output arrays
        trans_between_cap_times[:,:,s] = trans_times
        trans_between_cap_ratio[:,:,s] = trans_ratio
        #trans_between_cap_duration_mean[:,:,s] = trans_du_mean
        #trans_between_cap_duration_std[:,:,s] = trans_du_std

    return trans_between_cap_times, trans_between_cap_ratio

In [5]:
# Create sub_indices for transition analysis
sub_indices = []
label_info = []
current_idx = 0

for sub in subject_list:
    for ses in sessions:
        for run in runs:
            data_label = f"sub-{sub}_ses-{ses}_run-{run}"

            # Find this subject/session/run's data
            sub_mask = DATA_LABELS == data_label
            n_timepoints = np.sum(sub_mask)

            if n_timepoints > 0:
                sub_indices.append([current_idx, current_idx + n_timepoints])
                label_info.append({
                    'data_label': data_label,
                    'subject': sub,
                    'session': ses,
                    'run': run,
                    'idx': len(sub_indices) - 1
                })
                current_idx += n_timepoints

# Calculate transition probabilities
trans_times, trans_probs = do_transition_matrix(CAPtimeseries, sub_indices)

print(f"Transition times shape: {trans_times.shape}")
print(f"Transition probabilities shape: {trans_probs.shape}")

# Convert to DataFrame
transition_data = []

for idx, info in enumerate(label_info):
    # Create a record for each possible transition
    for i in range(k):
        for j in range(k):
            if i != j:  # Skip self-transitions
                record = {
                    'data_label': info['data_label'],
                    'subject': info['subject'],
                    'session': info['session'],
                    'session_name': sessions_dict.get(info['session'], f"Session_{info['session']}"),
                    'run': info['run'],
                    'run_name': runs_dict.get(info['run'], f"Run_{info['run']}"),
                    'from_CAP': f'CAP{i+1}',
                    'to_CAP': f'CAP{j+1}',
                    'transition_count': trans_times[i, j, idx],
                    'transition_probability': trans_probs[i, j, idx]
                }
                transition_data.append(record)

# Create DataFrame
transition_df = pd.DataFrame(transition_data)

# Save full transition data
output_file = os.path.join(out_dir, "CAP_transitions_full.csv")
transition_df.to_csv(output_file, index=False)

# Create average transition probability matrices for each session
for ses in sessions:
    ses_name = sessions_dict.get(ses, f"Session_{ses}")
    ses_data = transition_df[transition_df['session'] == ses]

    # Create mean transition probability matrix
    trans_matrix = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            if i != j:
                mean_prob = ses_data[
                    (ses_data['from_CAP'] == f'CAP{i+1}') &
                    (ses_data['to_CAP'] == f'CAP{j+1}')
                ]['transition_probability'].mean()
                trans_matrix[i, j] = mean_prob

    # Save matrix
    matrix_file = os.path.join(out_dir, f"transition_matrix_{ses_name}.npy")
    np.save(matrix_file, trans_matrix)

    # Also save as CSV for easy viewing
    matrix_df = pd.DataFrame(
        trans_matrix,
        index=[f'CAP{i+1}' for i in range(k)],
        columns=[f'CAP{i+1}' for i in range(k)]
    )
    csv_file = os.path.join(out_dir, f"transition_matrix_{ses_name}.csv")
    matrix_df.to_csv(csv_file)


Transition times shape: (7, 7, 60)
Transition probabilities shape: (7, 7, 60)
